# Task 2: Feature Engineering Challenge
Using a datetime-rich dataset (e.g., Uber rides, sales transactions, Airbnb listings), engineer at least 6 new features:
1. extract day-of-week, hour, is_weekend from timestamps
2. create 2 interaction features (multiply/ratio)
3. apply log transform to at least 1 skewed numeric column
4. bin one continuous variable
5. then compare model accuracy (use any simple classifier) before and after your engineered features
6. document the accuracy delta and explain why each feature help

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

### Load Dataset

In [2]:
df = pd.read_csv("retail_small.csv")

print(df.head())
print(df.shape)

  InvoiceNo StockCode                      Description  Quantity  \
0    540456     48185               DOORMAT FAIRY CAKE         2   
1    566891     23013    GLASS APOTHECARY BOTTLE TONIC         4   
2   C562139     21313      GLASS HEART T-LIGHT HOLDER         -4   
3    565438     22382       LUNCH BAG SPACEBOY DESIGN          4   
4    566016     21212  PACK OF 72 RETROSPOT CAKE CASES        24   

       InvoiceDate  UnitPrice  CustomerID         Country  
0   1/7/2011 12:14       7.95     13534.0  United Kingdom  
1  9/15/2011 13:51       3.95     14894.0  United Kingdom  
2   8/3/2011 10:10       0.85     12921.0  United Kingdom  
3   9/4/2011 13:56       1.65     17229.0  United Kingdom  
4   9/8/2011 12:20       0.55     15144.0  United Kingdom  
(10000, 8)


### Create Target Variable

Since the dataset has no target column, create one.

We will predict whether an order quantity is above the median.

In [3]:
median_qty = df["Quantity"].median()

df["HighQuantity"] = (
    df["Quantity"] > median_qty
).astype(int)

df["HighQuantity"].value_counts()

HighQuantity
0    5095
1    4905
Name: count, dtype: int64

### Convert Date Column

In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

### Baseline Model (Before Feature Engineering)

Use only original features.

In [5]:
baseline_df = df.copy()

# Select features:

X_base = baseline_df[
    [
        "StockCode",
        "Description",
        "Country",
        "UnitPrice"
    ]
]

y = baseline_df["HighQuantity"]

### Train-Test Split

In [6]:
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base,
    y,
    test_size=0.2,
    random_state=42
)

### Encode Categorical Features

In [7]:
cat_cols = [
    "StockCode",
    "Description",
    "Country"
]

num_cols = [
    "UnitPrice"
]

In [8]:
preprocessor_base = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        ),
        (
            "num",
            "passthrough",
            num_cols
        )
    ]
)

### Build Baseline Model

In [9]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_base),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [10]:
# Train:

baseline_model.fit(X_train_base, y_train)

# Predict:

y_pred_base = baseline_model.predict(X_test_base)

# Accuracy:

baseline_accuracy = accuracy_score(
    y_test,
    y_pred_base
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.6625


## Feature Engineering

In [11]:
# 1. Day of Week
df["day_of_week"] = df["InvoiceDate"].dt.dayofweek

# 2. Hour
df["hour"] = df["InvoiceDate"].dt.hour

# 3. Weekend Indicator
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

# 4. Interaction Feature: Total Amount
df["total_amount"] = (
    df["Quantity"] *
    df["UnitPrice"]
)

# 5. Interaction Feature: Price Ratio
df["price_quantity_ratio"] = (
    df["UnitPrice"] /
    (abs(df["Quantity"]) + 1)
)

# 6. Log Transform
df["log_unitprice"] = np.log1p(
    df["UnitPrice"]
)

# 7. Binning
df["price_bin"] = pd.cut(
    df["UnitPrice"],
    bins=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

# Check:

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,HighQuantity,day_of_week,hour,is_weekend,total_amount,price_quantity_ratio,log_unitprice,price_bin
0,540456,48185,DOORMAT FAIRY CAKE,2,2011-01-07 12:14:00,7.95,13534.0,United Kingdom,0,4,12,0,15.9,2.650,2.191654,Low
1,566891,23013,GLASS APOTHECARY BOTTLE TONIC,4,2011-09-15 13:51:00,3.95,14894.0,United Kingdom,0,3,13,0,15.8,0.790,1.599388,Low
2,C562139,21313,GLASS HEART T-LIGHT HOLDER,-4,2011-08-03 10:10:00,0.85,12921.0,United Kingdom,0,2,10,0,-3.4,0.170,0.615186,Low
3,565438,22382,LUNCH BAG SPACEBOY DESIGN,4,2011-09-04 13:56:00,1.65,17229.0,United Kingdom,0,6,13,1,6.6,0.330,0.974560,Low
4,566016,21212,PACK OF 72 RETROSPOT CAKE CASES,24,2011-09-08 12:20:00,0.55,15144.0,United Kingdom,1,3,12,0,13.2,0.022,0.438255,Low


### Create New Feature Set

In [12]:
X_eng = df[
    [
        "StockCode",
        "Description",
        "Country",
        "UnitPrice",
        "day_of_week",
        "hour",
        "is_weekend",
        "total_amount",
        "price_quantity_ratio",
        "log_unitprice",
        "price_bin"
    ]
]

y = df["HighQuantity"]

### Train-Test Split Again

In [13]:
X_train_eng, X_test_eng, y_train, y_test = train_test_split(
    X_eng,
    y,
    test_size=0.2,
    random_state=42
)

### Define Categorical and Numerical Columns

In [14]:
categorical_cols = [
    "StockCode",
    "Description",
    "Country",
    "price_bin"
]

numerical_cols = [
    "UnitPrice",
    "day_of_week",
    "hour",
    "is_weekend",
    "total_amount",
    "price_quantity_ratio",
    "log_unitprice"
]

### Preprocessing

In [15]:
preprocessor_eng = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            "passthrough",
            numerical_cols
        )
    ]
)

### Build Model

In [16]:
engineered_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_eng),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

# Train:

engineered_model.fit(
    X_train_eng,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['StockCode','Description','Country',...,'price_quantity_ratio', 'log_unitprice','price_bin']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``rema

### Prediction

In [18]:
y_pred_eng = engineered_model.predict(
    X_test_eng
)

# Accuracy:

engineered_accuracy = accuracy_score(
    y_test,
    y_pred_eng
)

print("Engineered Accuracy:", engineered_accuracy)

Engineered Accuracy: 0.999


### Accuracy Comparison

In [19]:
print("Baseline Accuracy :", baseline_accuracy)
print("Engineered Accuracy :", engineered_accuracy)

delta = engineered_accuracy - baseline_accuracy

print("Accuracy Improvement :", delta)

Baseline Accuracy : 0.6625
Engineered Accuracy : 0.999
Accuracy Improvement : 0.3365


### Final Report Table

In [20]:
results = pd.DataFrame(
    {
        "Model": [
            "Before Feature Engineering",
            "After Feature Engineering"
        ],
        "Accuracy": [
            baseline_accuracy,
            engineered_accuracy
        ]
    }
)

print(results)

                        Model  Accuracy
0  Before Feature Engineering    0.6625
1   After Feature Engineering    0.9990


### Assignment Write-up (Why Features Help)

| Feature              | Reason                                               |
| -------------------- | ---------------------------------------------------- |
| day_of_week          | Captures weekday shopping patterns                   |
| hour                 | Captures time-based purchasing behavior              |
| is_weekend           | Weekend customers behave differently                 |
| total_amount         | Measures overall transaction value                   |
| price_quantity_ratio | Shows relationship between price and quantity        |
| log_unitprice        | Reduces skewness and outlier impact                  |
| price_bin            | Converts continuous price into meaningful categories |
